# Prompt Evaluation Analysis

Loads the latest run under `results/prompt_eval/` and visualizes the model-level and scenario-level summaries for reasoning and interpretive benchmarks.

In [ ]:
from pathlib import Path

import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

RESULTS_ROOT = Path('..') / 'results' / 'prompt_eval'
run_dirs = sorted([path for path in RESULTS_ROOT.iterdir() if path.is_dir()])
if not run_dirs:
    raise FileNotFoundError('No prompt-eval runs found under results/prompt_eval')

RUN_DIR = run_dirs[-1]
print('Using run directory:', RUN_DIR)
config = json.loads((RUN_DIR / 'config.json').read_text(encoding='utf-8'))
config

In [ ]:
item_scores = pd.read_csv(RUN_DIR / 'item_scores.csv')
scenario_scores = pd.read_csv(RUN_DIR / 'scenario_scores.csv')
model_summary = pd.read_csv(RUN_DIR / 'model_summary.csv')

display(model_summary)
display(scenario_scores.head())
display(item_scores[['dataset', 'model', 'metric_id', 'item_id', 'primary_score']].head())

In [ ]:
interpretive = model_summary[model_summary['dataset'] == 'interpretive'].copy()
if not interpretive.empty:
    pivot = interpretive.pivot(index='model', columns='metric_id', values='primary_score')
    plt.figure(figsize=(10, 4))
    sns.heatmap(pivot, annot=True, cmap='YlGnBu', vmin=0, vmax=1)
    plt.title('Interpretive Proxy Scores by Model')
    plt.tight_layout()
    plt.show()
else:
    print('No interpretive rows in model_summary.csv')

In [ ]:
reasoning = model_summary[model_summary['dataset'].isin(['moralbench', 'morebench_public', 'morebench_theory'])].copy()
if not reasoning.empty:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=reasoning, x='metric_id', y='primary_score', hue='model')
    plt.xticks(rotation=30, ha='right')
    plt.ylim(0, 1)
    plt.title('Reasoning Benchmark Scores by Model')
    plt.tight_layout()
    plt.show()
else:
    print('No reasoning rows in model_summary.csv')

In [ ]:
examples_path = RUN_DIR / 'examples.jsonl'
if examples_path.exists():
    examples = pd.read_json(examples_path, lines=True)
    display(examples[['metric_id', 'model', 'item_id', 'primary_score', 'response_text']].head(10))
else:
    print('examples.jsonl not found for this run')